In [5]:
from glob import glob
import pandas as pd
import os
from tqdm import tqdm as tqdm 
from IPython.display import Image
from pathlib import Path
# import imageio
# import moviepy.video.io.ImageSequenceClip
from tqdm import tqdm
import h5py
import math
import matplotlib
# matplotlib.use('Agg') # Must be before importing matplotlib.pyplot or pylab!
import matplotlib.pyplot as plt
import copy
import numpy as np
import os
import json
import scipy.stats
import time
from types import SimpleNamespace
import random
import pandas as pd
from mpl_toolkits.axes_grid1 import make_axes_locatable


/afs/csail.mit.edu/u/c/czw/.config/matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /tmp/matplotlib-elee5_lt because there was an issue with the default path (/afs/csail.mit.edu/u/c/czw/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [6]:
results_root = "/storage/czw/neuroprobe/data/brainbert_results/brainbert_5s_WithinSession/linear_voltage_single_electrode/"


In [25]:
def get_test_results(results, eval_name):
    eval_results = results["evaluation_results"]
    assert len(eval_results.keys()) == 1
    records = []
    for subject_trial in eval_results:
        # print(subject_trial)
        subject = subject_trial.split("_")[0][len("btbank"):]
        trial = subject_trial.split("_")[1]
        subject_trial_results = eval_results[subject_trial]['electrode']
        for electrode in subject_trial_results:
            # print(electrode)
            
            electrode_results = subject_trial_results[electrode]
            time_bin = electrode_results["whole_window"]
            # for time_bin in time_bin_results:
            # print(time_bin)
            time_bin_start = time_bin['time_bin_start']
            fold_results = time_bin['folds']
            avg_test = np.mean([f['test_roc_auc'] for f in fold_results])
            records.append({
                "subject": subject,
                "trial": trial,
                "ID": f'{electrode}-sub_{subject}',
                "electrode": electrode,
                "avg_test": avg_test,
                "time_bin": time_bin_start,
                "task": eval_name
            })
        return records


In [26]:
all_records = []
all_results_paths = glob(os.path.join(results_root, "*"))
for path in tqdm(all_results_paths):
    name = Path(path).stem
    sub_id = name.split("_")[1][len("btbank"):]
    trial_id = name.split("_")[2]
    eval_name = "_".join(name.split("_")[3:])
    with open(path, "r") as f:
        results = json.load(f)
        test_results = get_test_results(results, eval_name)
    all_records += test_results
results_df = pd.DataFrame.from_records(all_records)



100%|██████████████████████| 14/14 [00:00<00:00, 186.46it/s]


In [27]:
results

{'model_name': 'Logistic Regression',
 'author': 'Andrii Zahorodnii',
 'description': 'Simple Logistic Regression.',
 'organization': 'MIT',
 'organization_url': 'https://mit.edu',
 'timestamp': 1764429780.490795,
 'evaluation_results': {'btbank7_0': {'electrode': {'LT1A1': {'time_bins': [],
     'whole_window': {'time_bin_start': -2.5,
      'time_bin_end': 2.5,
      'folds': [{'train_accuracy': 1.0,
        'train_roc_auc': 1.0,
        'test_accuracy': 0.5011428571428571,
        'test_roc_auc': 0.4966426122448979},
       {'train_accuracy': 1.0,
        'train_roc_auc': 1.0,
        'test_accuracy': 0.4937142857142857,
        'test_roc_auc': 0.4875226122448979}]}},
    'LT1A2': {'time_bins': [],
     'whole_window': {'time_bin_start': -2.5,
      'time_bin_end': 2.5,
      'folds': [{'train_accuracy': 1.0,
        'train_roc_auc': 1.0,
        'test_accuracy': 0.49942857142857144,
        'test_roc_auc': 0.4903549387755102},
       {'train_accuracy': 1.0,
        'train_roc_auc':

In [32]:
results_df[(results_df.task=="onset") & (results_df.avg_test>0.9)]

,subject,trial,ID,electrode,avg_test,time_bin,task
125,1,2,T1bIc6-sub_1,T1bIc6,0.902201,-2.5,onset
126,1,2,T1bIc7-sub_1,T1bIc7,0.946396,-2.5,onset
863,1,1,T1bIc7-sub_1,T1bIc7,0.906268,-2.5,onset
983,3,0,T1cIe11-sub_3,T1cIe11,0.915334,-2.5,onset
985,3,0,T1b1-sub_3,T1b1,0.925877,-2.5,onset
1212,3,1,T1cIe11-sub_3,T1cIe11,0.926394,-2.5,onset
1214,3,1,T1b1-sub_3,T1b1,0.913742,-2.5,onset
